# CS 131 — Wildfire Burn Severity Mapping
Set runtime to **T4 GPU** before running: `Runtime → Change runtime type → T4 GPU`

**If restarting after a crash:** set `SKIP_PREPROCESS = True` in the Setup cell below to restore processed files from Drive instead of rerunning SIFT alignment.

In [ ]:
!pip install -q rasterio scikit-image tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
DRIVE_RAW  = '/content/drive/MyDrive/cs131_wildfire'
DRIVE_PROC = f'{DRIVE_RAW}/processed'   # saved after first run
DRIVE_OUT  = f'{DRIVE_RAW}/outputs'
LOCAL_RAW  = '/content/data/raw'
LOCAL_PROC = '/content/data/processed'
LOCAL_OUT  = '/content/outputs'

for d in [LOCAL_RAW, LOCAL_PROC, LOCAL_OUT, DRIVE_OUT, DRIVE_PROC]:
    os.makedirs(d, exist_ok=True)

# Set True if processed/ folder already exists in Drive from a prior run
SKIP_PREPROCESS = os.path.exists(f'{DRIVE_PROC}/camp_fire')
print(f'SKIP_PREPROCESS = {SKIP_PREPROCESS}')

if SKIP_PREPROCESS:
    print('Restoring processed TIFFs + outputs from Drive...')
    shutil.copytree(DRIVE_PROC, LOCAL_PROC, dirs_exist_ok=True)
    shutil.copytree(DRIVE_OUT,  LOCAL_OUT,  dirs_exist_ok=True)
    print('Restored:', os.listdir(LOCAL_PROC))
else:
    # copy raw tiffs
    copied = []
    for f in os.listdir(DRIVE_RAW):
        if f.endswith('.tif') and '(' not in f:   # skip duplicate (1) copies
            shutil.copy(f'{DRIVE_RAW}/{f}', f'{LOCAL_RAW}/{f}')
            copied.append(f)
    print(f'Copied {len(copied)} raw TIFFs:', sorted(copied))

In [ ]:
import numpy as np
import cv2
import rasterio
import gc
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from skimage.filters import gaussian, threshold_multiotsu
from skimage.feature import canny
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

FIRES = [
    ('camp_fire',       2018),
    ('dixie_fire',      2021),
    ('caldor_fire',     2021),
    ('bootleg_fire',    2021),
    ('august_complex',  2020),
]

CLASS_LABELS = ['unburned', 'low', 'moderate', 'high']
CLASS_COLORS = ['#2ecc71', '#f1c40f', '#e67e22', '#c0392b']
DNBR_THRESHOLDS = [0.1, 0.27, 0.44]
CMAP = mcolors.ListedColormap(CLASS_COLORS)
NORM = mcolors.BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], CMAP.N)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
def load_tiff(path):
    with rasterio.open(path) as src:
        data = src.read().astype(np.float32)
        profile = src.profile
    return np.transpose(data, (1, 2, 0)), profile

def normalize_band(band):
    band = np.nan_to_num(band, nan=0.0)
    lo, hi = np.percentile(band, 2), np.percentile(band, 98)
    return np.clip((band - lo) / (hi - lo + 1e-8), 0, 1)

def normalize_img(img):
    return np.stack([normalize_band(img[:, :, c]) for c in range(img.shape[2])], axis=2)

def to_uint8(band):
    return (np.nan_to_num(band) * 255).astype(np.uint8)

def sift_align(pre_img, post_img):
    pre_gray  = to_uint8(normalize_band(pre_img[:, :, 2]))
    post_gray = to_uint8(normalize_band(post_img[:, :, 2]))
    sift = cv2.SIFT_create(nfeatures=5000)
    kp1, des1 = sift.detectAndCompute(pre_gray, None)
    kp2, des2 = sift.detectAndCompute(post_gray, None)
    bf   = cv2.BFMatcher(cv2.NORM_L2)
    raw  = bf.knnMatch(des1, des2, k=2)
    good = [m for m, n in raw if m.distance < 0.75 * n.distance]
    print(f'  SIFT: {len(kp1)}/{len(kp2)} keypoints, {len(good)} good matches')
    if len(good) < 10:
        print('  too few matches, skipping alignment')
        return post_img
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    H, mask = cv2.findHomography(dst_pts, src_pts, cv2.RANSAC, 5.0)
    print(f'  inliers: {int(mask.sum())}/{len(good)}')
    h, w = pre_img.shape[:2]
    return np.stack([
        cv2.warpPerspective(post_img[:, :, c], H, (w, h))
        for c in range(post_img.shape[2])
    ], axis=2)

def save_tiff(array, profile, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    p = profile.copy()
    p.update(count=array.shape[2], dtype=rasterio.float32)
    with rasterio.open(path, 'w', **p) as dst:
        dst.write(np.transpose(array, (2, 0, 1)))

def compute_nbr(img):
    nir   = np.nan_to_num(img[:, :, 3], nan=0.0)
    swir2 = np.nan_to_num(img[:, :, 5], nan=0.0)
    denom = nir + swir2
    return np.where(denom != 0, (nir - swir2) / denom, 0.0)

def dnbr_to_classes(dnbr):
    labels = np.zeros(dnbr.shape, dtype=np.int32)
    labels[dnbr > DNBR_THRESHOLDS[0]] = 1
    labels[dnbr > DNBR_THRESHOLDS[1]] = 2
    labels[dnbr > DNBR_THRESHOLDS[2]] = 3
    return labels

def compute_iou(pred, gt):
    pred, gt = pred.ravel(), gt.ravel()
    ious = {}
    for i, cls in enumerate(CLASS_LABELS):
        inter = ((pred == i) & (gt == i)).sum()
        union = ((pred == i) | (gt == i)).sum()
        ious[cls] = float(inter) / float(union + 1e-8)
    ious['mIoU'] = float(np.mean(list(ious.values())))
    return ious

def pixel_acc(pred, gt):
    return float((pred.ravel() == gt.ravel()).mean())

## 1. Preprocessing (skipped if SKIP_PREPROCESS=True)

In [ ]:
available_fires = []

if SKIP_PREPROCESS:
    for name, year in FIRES:
        if os.path.exists(f'{LOCAL_PROC}/{name}/pre.tif'):
            available_fires.append((name, year))
    print('Restored fires:', [n for n, _ in available_fires])
else:
    for name, year in FIRES:
        pre_path  = f'{LOCAL_RAW}/{name}_pre_{year}.tif'
        post_path = f'{LOCAL_RAW}/{name}_post_{year}.tif'
        if not (os.path.exists(pre_path) and os.path.exists(post_path)):
            print(f'SKIP {name}: not found'); continue

        print(f'\n--- {name} {year} ---')
        pre_img,  profile = load_tiff(pre_path)
        post_img, _       = load_tiff(post_path)
        print(f'  shape: {pre_img.shape}')

        post_aligned = sift_align(pre_img, post_img);  del post_img
        pre_norm     = normalize_img(pre_img);          del pre_img
        post_norm    = normalize_img(post_aligned);     del post_aligned

        out_dir = f'{LOCAL_PROC}/{name}'
        os.makedirs(out_dir, exist_ok=True)
        save_tiff(pre_norm,  profile, f'{out_dir}/pre.tif')
        save_tiff(post_norm, profile, f'{out_dir}/post.tif')

        dnbr   = compute_nbr(pre_norm) - compute_nbr(post_norm)
        labels = dnbr_to_classes(dnbr)
        del pre_norm, post_norm

        lbl_dir = f'{LOCAL_OUT}/{name}'
        os.makedirs(lbl_dir, exist_ok=True)
        lp = profile.copy(); lp.update(count=1, dtype=rasterio.int32)
        with rasterio.open(f'{lbl_dir}/labels_usgs.tif', 'w', **lp) as dst:
            dst.write(labels[np.newaxis])
        sp = profile.copy(); sp.update(count=1, dtype=rasterio.float32)
        with rasterio.open(f'{lbl_dir}/dnbr.tif', 'w', **sp) as dst:
            dst.write(dnbr[np.newaxis])

        counts = [(labels == i).mean() * 100 for i in range(4)]
        print(f'  dNBR [{dnbr.min():.3f}, {dnbr.max():.3f}]  |  '
              + '  '.join(f'{c}: {v:.1f}%' for c, v in zip(CLASS_LABELS, counts)))
        del dnbr, labels
        gc.collect()
        available_fires.append((name, year))

    print('\nSaving processed TIFFs to Drive...')
    shutil.copytree(LOCAL_PROC, DRIVE_PROC, dirs_exist_ok=True)
    print('Done.')

fire_names = [n for n, _ in available_fires]
print(f'\nReady: {fire_names}')

## 2. Classical Pipeline

In [ ]:
def run_classical(dnbr, sigma_gauss=1.5, sigma_canny=1.0):
    dnbr_clean = np.nan_to_num(dnbr, nan=0.0)
    smooth = gaussian(dnbr_clean, sigma=sigma_gauss)
    d_min, d_max = smooth.min(), smooth.max()
    norm_d = (smooth - d_min) / (d_max - d_min + 1e-8)
    edges  = canny(norm_d, sigma=sigma_canny, low_threshold=0.05, high_threshold=0.15)
    thresholds = threshold_multiotsu(smooth, classes=4)
    pred = np.digitize(smooth, bins=thresholds).astype(np.int32)
    return smooth, edges, pred, thresholds

classical_metrics = {}

for name in fire_names:
    print(f'\n--- Classical: {name} ---')
    with rasterio.open(f'{LOCAL_OUT}/{name}/dnbr.tif') as src:
        dnbr = src.read(1).astype(np.float32)
    with rasterio.open(f'{LOCAL_OUT}/{name}/labels_usgs.tif') as src:
        gt = src.read(1).astype(np.int32)

    smooth, edges, pred, thresh = run_classical(dnbr)
    ious = compute_iou(pred, gt)
    acc  = pixel_acc(pred, gt)
    classical_metrics[name] = dict(ious=ious, acc=acc)
    print(f'  mIoU={ious["mIoU"]:.4f}  PixAcc={acc:.4f}  Otsu={np.round(thresh, 3)}')

    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    fig.suptitle(f"{name.replace('_',' ').title()} — Classical Pipeline  mIoU={ious['mIoU']:.3f}", fontweight='bold')
    vmax = max(abs(np.nanmin(dnbr)), abs(np.nanmax(dnbr)))
    axes[0].imshow(np.nan_to_num(dnbr),   cmap='RdYlGn_r', vmin=-vmax, vmax=vmax); axes[0].set_title('Raw dNBR');    axes[0].axis('off')
    axes[1].imshow(smooth, cmap='RdYlGn_r', vmin=-vmax, vmax=vmax);                axes[1].set_title('Smoothed');    axes[1].axis('off')
    axes[2].imshow(edges,  cmap='gray');                                            axes[2].set_title('Canny Edges'); axes[2].axis('off')
    axes[3].imshow(pred, cmap=CMAP, norm=NORM);                                     axes[3].set_title('Otsu Pred');  axes[3].axis('off')
    im = axes[4].imshow(gt, cmap=CMAP, norm=NORM);                                  axes[4].set_title('USGS GT');    axes[4].axis('off')
    cbar = plt.colorbar(im, ax=axes[4], fraction=0.046, ticks=[0,1,2,3])
    cbar.ax.set_yticklabels(CLASS_LABELS)
    iou_str = '  '.join(f'{c}: {ious[c]:.3f}' for c in CLASS_LABELS)
    fig.text(0.01, 0.01, f'IoU — {iou_str}', fontsize=8, va='bottom', family='monospace')
    plt.tight_layout()
    plt.savefig(f'{LOCAL_OUT}/{name}/classical_pipeline.png', dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    del dnbr, gt, smooth, edges, pred
    gc.collect()

## 3. U-Net

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels=12, num_classes=4, features=[32, 64, 128, 256]):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups   = nn.ModuleList()
        self.pool  = nn.MaxPool2d(2, 2)
        ch = in_channels
        for f in features:
            self.downs.append(DoubleConv(ch, f)); ch = f
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)
        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(f * 2, f, 2, 2))
            self.ups.append(DoubleConv(f * 2, f))
        self.head = nn.Conv2d(features[0], num_classes, 1)

    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x); skips.append(x); x = self.pool(x)
        x = self.bottleneck(x)
        skips = skips[::-1]
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            s = skips[i // 2]
            if x.shape != s.shape:
                x = F.interpolate(x, size=s.shape[2:])
            x = torch.cat([s, x], dim=1)
            x = self.ups[i + 1](x)
        return self.head(x)


class WildfireDataset(Dataset):
    def __init__(self, fire_names, proc_dir, out_dir,
                 patch_size=256, stride=128, split='train', val_ratio=0.15, augment=False):
        self.proc_dir, self.out_dir = proc_dir, out_dir
        self.patch_size = patch_size
        self.augment    = augment
        coords = []
        for name in fire_names:
            with rasterio.open(f'{proc_dir}/{name}/pre.tif') as src:
                H, W = src.height, src.width
            for y in range(0, H - patch_size + 1, stride):
                for x in range(0, W - patch_size + 1, stride):
                    coords.append((name, y, x))
        np.random.seed(42)
        idx = np.random.permutation(len(coords))
        cut = int(len(idx) * (1 - val_ratio))
        chosen = idx[:cut] if split == 'train' else idx[cut:]
        self.coords = [coords[i] for i in chosen]
        print(f'  {split}: {len(self.coords)} patches')

    def __len__(self): return len(self.coords)

    def __getitem__(self, idx):
        name, row, col = self.coords[idx]
        p   = self.patch_size
        win = rasterio.windows.Window(col, row, p, p)
        with rasterio.open(f'{self.proc_dir}/{name}/pre.tif') as src:
            pre  = src.read(window=win).astype(np.float32)
        with rasterio.open(f'{self.proc_dir}/{name}/post.tif') as src:
            post = src.read(window=win).astype(np.float32)
        with rasterio.open(f'{self.out_dir}/{name}/labels_usgs.tif') as src:
            lbl  = src.read(1, window=win).astype(np.int64)

        img = np.nan_to_num(np.concatenate([pre, post], axis=0), nan=0.0)
        if self.augment:
            if np.random.rand() < 0.5:
                img = np.flip(img, axis=2).copy(); lbl = np.flip(lbl, axis=1).copy()
            if np.random.rand() < 0.5:
                img = np.flip(img, axis=1).copy(); lbl = np.flip(lbl, axis=0).copy()
        return torch.from_numpy(img), torch.from_numpy(lbl)

In [ ]:
PATCH_SIZE = 256
STRIDE     = 128
EPOCHS     = 30
LR         = 1e-3
BATCH_SIZE = 8

# class weights to handle severe unburned-class imbalance
CLASS_WEIGHTS = torch.tensor([0.3, 1.0, 2.0, 1.5], dtype=torch.float32).to(DEVICE)

print('Building datasets...')
train_ds = WildfireDataset(fire_names, LOCAL_PROC, LOCAL_OUT, patch_size=PATCH_SIZE, stride=STRIDE, split='train', augment=True)
val_ds   = WildfireDataset(fire_names, LOCAL_PROC, LOCAL_OUT, patch_size=PATCH_SIZE, stride=STRIDE, split='val',   augment=False)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

model     = UNet(in_channels=12, num_classes=4).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)

history = {'train_loss': [], 'val_loss': [], 'val_miou': []}

print(f'\nTraining on {DEVICE} for {EPOCHS} epochs...')
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for imgs, lbls in tqdm(train_dl, desc=f'Epoch {epoch+1}/{EPOCHS}', leave=False):
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        loss = criterion(model(imgs), lbls)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0
    all_pred, all_lbl = [], []
    with torch.no_grad():
        for imgs, lbls in val_dl:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            out = model(imgs)
            val_loss += criterion(out, lbls).item()
            all_pred.append(out.argmax(1).cpu().numpy())
            all_lbl.append(lbls.cpu().numpy())

    pf   = np.concatenate([p.ravel() for p in all_pred])
    lf   = np.concatenate([l.ravel() for l in all_lbl])
    ious = compute_iou(pf, lf)
    tl   = train_loss / len(train_dl)
    vl   = val_loss   / len(val_dl)
    scheduler.step()
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['val_miou'].append(ious['mIoU'])
    print(f'Epoch {epoch+1:>2}: train={tl:.4f}  val={vl:.4f}  mIoU={ious["mIoU"]:.4f}')

torch.save(model.state_dict(), f'{LOCAL_OUT}/unet_weights.pt')
print('Weights saved.')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='train'); ax1.plot(history['val_loss'], label='val')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()
ax2.plot(history['val_miou'], color='green')
ax2.set_title('Val mIoU'); ax2.set_xlabel('Epoch')
plt.suptitle('U-Net Training Curves', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{LOCAL_OUT}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

## 4. U-Net Inference

In [ ]:
def predict_fire(model, name, proc_dir, patch_size=256, stride=64, device='cuda'):
    model.eval()
    with rasterio.open(f'{proc_dir}/{name}/pre.tif') as src:
        H, W = src.height, src.width
    logits_sum = np.zeros((4, H, W), dtype=np.float32)
    count_map  = np.zeros((H, W),    dtype=np.float32)
    with torch.no_grad():
        for row in range(0, H - patch_size + 1, stride):
            for col in range(0, W - patch_size + 1, stride):
                win = rasterio.windows.Window(col, row, patch_size, patch_size)
                with rasterio.open(f'{proc_dir}/{name}/pre.tif') as src:
                    pre  = src.read(window=win).astype(np.float32)
                with rasterio.open(f'{proc_dir}/{name}/post.tif') as src:
                    post = src.read(window=win).astype(np.float32)
                img = np.nan_to_num(np.concatenate([pre, post], axis=0), nan=0.0)
                t   = torch.from_numpy(img).unsqueeze(0).to(device)
                out = model(t).squeeze(0).cpu().numpy()
                logits_sum[:, row:row+patch_size, col:col+patch_size] += out
                count_map[row:row+patch_size, col:col+patch_size]     += 1
    count_map = np.maximum(count_map, 1)
    return np.argmax(logits_sum / count_map[np.newaxis], axis=0).astype(np.int32)

unet_metrics = {}

for name in fire_names:
    print(f'\n--- U-Net: {name} ---')
    pred = predict_fire(model, name, LOCAL_PROC, patch_size=PATCH_SIZE, stride=64, device=DEVICE)
    with rasterio.open(f'{LOCAL_OUT}/{name}/labels_usgs.tif') as src:
        gt = src.read(1).astype(np.int32)
    ious = compute_iou(pred, gt)
    acc  = pixel_acc(pred, gt)
    unet_metrics[name] = dict(ious=ious, acc=acc)
    print(f'  mIoU={ious["mIoU"]:.4f}  PixAcc={acc:.4f}')

    with rasterio.open(f'{LOCAL_PROC}/{name}/post.tif') as src:
        post_bands = src.read([3, 2, 1]).astype(np.float32)
    rgb = np.clip(np.nan_to_num(np.transpose(post_bands, (1,2,0))), 0, 1)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(f"{name.replace('_',' ').title()} — U-Net vs Classical", fontweight='bold')
    axes[0].imshow(rgb);                        axes[0].set_title('Post-fire RGB');  axes[0].axis('off')
    axes[1].imshow(gt,   cmap=CMAP, norm=NORM); axes[1].set_title('USGS GT');        axes[1].axis('off')
    axes[2].imshow(gt,   cmap=CMAP, norm=NORM)
    axes[2].set_title(f'Classical mIoU={classical_metrics[name]["ious"]["mIoU"]:.3f}'); axes[2].axis('off')
    im = axes[3].imshow(pred, cmap=CMAP, norm=NORM)
    axes[3].set_title(f'U-Net mIoU={ious["mIoU"]:.3f}'); axes[3].axis('off')
    cbar = plt.colorbar(im, ax=axes[3], fraction=0.046, ticks=[0,1,2,3])
    cbar.ax.set_yticklabels(CLASS_LABELS)
    plt.tight_layout()
    plt.savefig(f'{LOCAL_OUT}/{name}/unet_vs_classical.png', dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    del pred, gt, rgb, post_bands
    gc.collect()

## 5. Summary

In [ ]:
header = ['unburned', 'low', 'moderate', 'high', 'mIoU', 'PixAcc']
print(f'{"Fire":<20}  {"Method":<12}  ' + '  '.join(f'{c:<10}' for c in header))
print('=' * 95)
for name in fire_names:
    for label, res in [('Classical', classical_metrics[name]), ('U-Net', unet_metrics[name])]:
        vals = [f'{res["ious"][c]:.4f}' for c in CLASS_LABELS]
        vals += [f'{res["ious"]["mIoU"]:.4f}', f'{res["acc"]:.4f}']
        print(f'{name:<20}  {label:<12}  ' + '  '.join(f'{v:<10}' for v in vals))
    print('-' * 95)

print('\n=== AVERAGES ===')
for label, metrics in [('Classical', classical_metrics), ('U-Net', unet_metrics)]:
    mious = [metrics[n]['ious']['mIoU'] for n in fire_names]
    accs  = [metrics[n]['acc']          for n in fire_names]
    print(f'{label}: avg mIoU={np.mean(mious):.4f}±{np.std(mious):.4f}  avg PixAcc={np.mean(accs):.4f}±{np.std(accs):.4f}')

In [ ]:
x = np.arange(len(fire_names)); w = 0.35
c_mious = [classical_metrics[n]['ious']['mIoU'] for n in fire_names]
u_mious = [unet_metrics[n]['ious']['mIoU']      for n in fire_names]

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - w/2, c_mious, w, label='Classical (Gaussian→Canny→Otsu)', color='steelblue',  alpha=0.85)
b2 = ax.bar(x + w/2, u_mious, w, label='U-Net (12-band pre+post)',         color='darkorange', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([n.replace('_',' ').title() for n in fire_names], rotation=15, ha='right')
ax.set_ylabel('mIoU'); ax.set_ylim(0, 1)
ax.set_title('Burn Severity mIoU — Classical vs U-Net', fontweight='bold')
ax.legend()
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(f'{LOCAL_OUT}/comparison_miou.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

# per-class
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Per-Class IoU — All Fires', fontweight='bold')
xpos = np.arange(len(fire_names))
short = [n.split('_')[0].capitalize() for n in fire_names]
for i, cls in enumerate(CLASS_LABELS):
    cv = [classical_metrics[n]['ious'][cls] for n in fire_names]
    uv = [unet_metrics[n]['ious'][cls]      for n in fire_names]
    axes[i].bar(xpos - 0.2, cv, 0.4, label='Classical', color='steelblue',  alpha=0.85)
    axes[i].bar(xpos + 0.2, uv, 0.4, label='U-Net',     color='darkorange', alpha=0.85)
    axes[i].set_title(cls.capitalize())
    axes[i].set_xticks(xpos)
    axes[i].set_xticklabels(short, rotation=30, ha='right')
    axes[i].set_ylim(0, 1)
    if i == 0: axes[i].legend(fontsize=8)
plt.tight_layout()
plt.savefig(f'{LOCAL_OUT}/comparison_per_class.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

In [ ]:
shutil.copytree(LOCAL_OUT, DRIVE_OUT, dirs_exist_ok=True)
print('All outputs saved to Drive.')
print('Files:', sorted(os.listdir(DRIVE_OUT)))